# 02 · Download validation data for the season dates

Independent dates to check the Sentinel-2 season dates of `04` against: European larch *needle colouring 50 %* and *needle drop 50 %*, observed by eye near the AOI. Colouring 50 % corresponds to the yellowing peak / midpoint, needle drop 50 % to the end of the season. Two separate sources, kept in separate files and compared separately in `04`:

- **MeteoSwiss phenology network** (open data, no account): stations within `VALIDATION_RADIUS_KM` of the AOI centre. They lie lower than most of the AOI, so `04` compares year-to-year anomalies, not absolute dates.
- **Local larch observations** (`LOCAL_PHENO_FILE`, `Lärchen_pheno.xlsx`): sites closer to the AOI, same phases.

- **Settings:** `config.py` (`VALIDATION_RADIUS_KM`, `PHENO_PARAMS`, `LOCAL_PHENO_FILE`, `REFERENCE_FOREST_FILE`, years)
- **Reads:** `cache/<AOI>_zones/` (`zones.tif` and, for the autumn image, the statistics, from `01_download_sentinel2`), `LOCAL_PHENO_FILE`, `REFERENCE_FOREST_FILE`
- **Writes** to `cache/validation/<AOI>/`:
  - `meteoswiss_larch.csv`, `local_larch.csv`: one row per station / site, year and phase
  - `meteoswiss_stations.csv`, `local_sites.csv`: the stations and sites used
  - `station_map.png`, `aoi_map.png`: maps of the stations and sites and of the AOI with the Swiss National Park
  - `aoi_reference_map.png`, `aoi_imagery_map.png`, `aoi_forest_types.csv`: checks of the larch classification (AOI) against a forest-type map and against images
- **Next:** `04_phenology_metrics` (section 7 compares both methods with these data)

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import requests
import rioxarray
import stackstac
import xarray as xr
from matplotlib import patheffects
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from pyproj import Geod, Transformer
from rasterio.features import shapes
from shapely.geometry import shape

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import clean_frame, collection_asset, download, last_complete_season, load_stats, stac_items

PHENO_COLLECTION = "ch.meteoschweiz.ogd-phenology"
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

zones_tif = rioxarray.open_rasterio(CACHE_DIR / "zones.tif", masked=True)
x0, y0, x1, y1 = zones_tif.rio.bounds()
to_wgs84 = Transformer.from_crs(zones_tif.rio.crs, "EPSG:4326", always_xy=True)
AOI_LON, AOI_LAT = to_wgs84.transform((x0 + x1) / 2, (y0 + y1) / 2)
print(f"AOI: {AOI_NAME} | centre {AOI_LAT:.3f} N, {AOI_LON:.3f} E | writes to {VALIDATION_DIR}")

## MeteoSwiss phenology observations (European larch)

The Swiss Phenology Network (about 160 stations) records the date when 50 % of the larch needles at a site are coloured (`mlard94d`) or have fallen (`mlard95d`). Data: MeteoSwiss open data, collection `ch.meteoschweiz.ogd-phenology` (one CSV per station, one column per phase, dates as YYYYMMDD).

Stations within `VALIDATION_RADIUS_KM` of the AOI centre that observe one of `PHENO_PARAMS` are kept.

In [ ]:
def distance_km(lat, lon):
    # Distance to the AOI centre in km (equirectangular approximation)
    return 6371 * np.hypot(np.radians(lat - AOI_LAT), np.radians(lon - AOI_LON) * np.cos(np.radians(AOI_LAT)))


def count_years(sites, obs):
    # Years since FIRST_YEAR with a date, per station and phase; stations without any are dropped
    sites = sites.join(obs[obs.year >= FIRST_YEAR].pivot_table(index="station", columns="param", values="year",
                                                                aggfunc="nunique").add_prefix("years_"))
    return sites.dropna(how="all", subset=[c for c in sites if c.startswith("years_")])


def plot_years(obs, sites, source):
    # Day of year per year since FIRST_YEAR, one line per station (seasons still running are left out)
    fig, axes = plt.subplots(1, len(PHENO_PARAMS), figsize=(12, 4), sharey=True)
    for ax, (param, phase) in zip(axes, PHENO_PARAMS.items()):
        for abbr, g in obs[(obs.param == param) & obs.year.between(FIRST_YEAR, last_complete_season())].groupby("station"):
            ax.plot(g.year, g.doy, "o-", ms=4, label=f"{abbr} ({sites.height_masl[abbr]:.0f} m)")
        ax.set_title(f"{source}: larch {phase}")
        ax.set_xlabel("year")
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("day of year")
    axes[-1].legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()


def read_meta(name):
    path = download(collection_asset(PHENO_COLLECTION, name), VALIDATION_DIR / "meteoswiss" / name)
    return pd.read_csv(path, sep=";", encoding="cp1252")


stations = read_meta("ogd-phenology_meta_stations.csv").set_index("station_abbr")
inventory = read_meta("ogd-phenology_meta_datainventory.csv")
observing = inventory[inventory.parameter_shortname.isin(PHENO_PARAMS)].station_abbr.unique()
stations["distance_km"] = distance_km(stations.station_coordinates_wgs84_lat, stations.station_coordinates_wgs84_lon)
near = stations.loc[observing].query("distance_km <= @VALIDATION_RADIUS_KM").sort_values("distance_km")
near = near[["station_name", "station_height_masl", "distance_km", "station_coordinates_wgs84_lat",
             "station_coordinates_wgs84_lon"]].rename(columns=lambda c: c.replace("station_", "").replace("coordinates_wgs84_", ""))
print(f"{len(near)} stations within {VALIDATION_RADIUS_KM} km observe larch needle colouring or drop")

# One CSV per station (STAC item id = lower-case abbreviation)
assets = {item["id"].upper(): next(iter(item["assets"].values()))["href"] for item in stac_items(PHENO_COLLECTION)}
rows = []
for abbr in near.index:
    path = download(assets[abbr], VALIDATION_DIR / "meteoswiss" / f"{abbr.lower()}_y.csv")
    data = pd.read_csv(path, sep=";", encoding="cp1252", dtype=str)
    year = pd.to_datetime(data.reference_timestamp, format="%d.%m.%Y %H:%M").dt.year
    for param, phase in PHENO_PARAMS.items():
        if param not in data:
            continue
        when = pd.to_datetime(data[param], format="%Y%m%d", errors="coerce")
        ok = when.notna()
        rows.append(pd.DataFrame({"station": abbr, "year": year[ok], "param": param, "phase": phase,
                                  "date": when[ok].dt.date, "doy": when[ok].dt.dayofyear}))
larch_obs = pd.concat(rows, ignore_index=True).join(near[["name", "height_masl", "distance_km"]], on="station")
larch_obs = larch_obs[larch_obs.year.between(FIRST_YEAR - 10, LAST_YEAR)]   # some years before for the station means
near = count_years(near, larch_obs)
display(near.round(1))

larch_obs.to_csv(VALIDATION_DIR / "meteoswiss_larch.csv", index=False)
near.to_csv(VALIDATION_DIR / "meteoswiss_stations.csv")
plot_years(larch_obs, near, "MeteoSwiss")

## Local larch observations (`Lärchen_pheno.xlsx`)

A second, separate source: larch observations at sites near the AOI (file `LOCAL_PHENO_FILE`, one row per site, year and phase; site code in `site_region`, LV95 coordinates, date in `pheno_date`). The same phases as MeteoSwiss (`PHENO_PARAMS`) are kept; leaf unfolding is in the file as well but not used, as `04` covers the autumn only.

- Larch rows (`larix_decidua`) with a date; sites within `VALIDATION_RADIUS_KM` of the AOI centre.
- **Names:** the file has only site codes; `SITE_NAMES` gives the place each code stands for (swisstopo place names at the site's coordinates).
- **Elevation:** the file has none; it is read from the swisstopo height service at each site's coordinates.
- **Observer notes:** the anomaly codes and remarks of the file are kept (`obs_anomalies`, `obs_remarks`). `quantity` says whether a site is a single tree (`individual`) or a `stand`.
- **Overlap:** site `lue` (Lü) is also the MeteoSwiss station `LLU`. The sources stay separate, so Lü is part of both.

In [ ]:
HEIGHT_SERVICE = "https://api3.geo.admin.ch/rest/services/height"   # swisstopo terrain height at an LV95 point
SITE_NAMES = {"clu": "Val Cluozza", "fuo": "Il Fuorn", "min": "Val Mingèr", "tru": "Val Trupchun", "lue": "Lü"}

raw = pd.read_excel(LOCAL_PHENO_FILE)
raw = raw[(raw.site_species_lat == "larix_decidua") & raw.param_shortname.isin(PHENO_PARAMS) & raw.pheno_date.notna()]
sites = raw.groupby("site_region").agg(quantity=("site_quantity", "first"), x=("site_x_lv95", "first"),
                                       y=("site_y_lv95", "first")).rename_axis("station")
lv95_to_wgs84 = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True)
sites["lon"], sites["lat"] = lv95_to_wgs84.transform(sites.x.values, sites.y.values)
sites["height_masl"] = [float(requests.get(HEIGHT_SERVICE, params={"easting": x, "northing": y, "sr": 2056},
                                           timeout=30).json()["height"]) for x, y in zip(sites.x, sites.y)]
sites["distance_km"] = distance_km(sites.lat, sites.lon)
sites["name"] = sites.index.map(lambda code: SITE_NAMES.get(code, code))
local_sites = sites.query("distance_km <= @VALIDATION_RADIUS_KM").sort_values("distance_km")
local_sites = local_sites[["name", "quantity", "height_masl", "distance_km", "lat", "lon"]]
print(f"{len(local_sites)} local sites within {VALIDATION_RADIUS_KM} km observe larch needle colouring or drop")

when = raw.pheno_date
local_obs = pd.DataFrame({"station": raw.site_region, "year": when.dt.year, "param": raw.param_shortname,
                          "phase": raw.param_shortname.map(PHENO_PARAMS), "date": when.dt.date, "doy": when.dt.dayofyear,
                          "obs_anomalies": raw.obs_anomalies, "obs_remarks": raw.obs_remarks})
local_obs = local_obs[local_obs.station.isin(local_sites.index) & local_obs.year.between(FIRST_YEAR - 10, LAST_YEAR)]
local_obs = (local_obs.join(local_sites[["name", "height_masl", "distance_km"]], on="station")
             .sort_values(["station", "param", "year"], ignore_index=True))
local_sites = count_years(local_sites, local_obs)
display(local_sites.round(1))

local_obs.to_csv(VALIDATION_DIR / "local_larch.csv", index=False)
local_sites.to_csv(VALIDATION_DIR / "local_sites.csv")
plot_years(local_obs, local_sites, "Local sites")

## Maps

Two maps for presentations, on the swisstopo national map (grey), saved at 300 dpi:

- `station_map.png`: the MeteoSwiss stations (blue dots), the local sites (purple diamonds), the AOI's larch pixels and the Swiss National Park, with the search radius `VALIDATION_RADIUS_KM` around the AOI centre
- `aoi_map.png`: the AOI's larch pixels and the border of the Swiss National Park (BAFU, collection `ch.bafu.schutzgebiete-schweizerischer_nationalpark`)

Map tiles and the park border are cached in `cache/maps/`.

In [ ]:
MAP_TILES = "https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.pixelkarte-grau/default/current/3857/{z}/{x}/{y}.jpeg"
MAP_CACHE = ROOT / "cache" / "maps"
ORIGIN = np.pi * 6378137   # m, half the circumference of the Web Mercator (EPSG:3857) sphere
LARCH_COLOR = "#d95f02"
STATION_COLOR = "#1b4f8a"   # MeteoSwiss stations
LOCAL_COLOR = "#7b3294"     # local sites
# Label offset in points (x, y) where labels would overlap; default (6, 4) = above right, x < 0: left, y < 0: below
LABEL_OFFSET = {"SMO": (-6, 4), "ZUO": (-6, 4), "LLU": (-6, -6)}


def basemap(ax, xmin, ymin, xmax, ymax, zoom, url=MAP_TILES, alpha=0.6):
    # swisstopo tiles of the WMTS layer in url covering a box in EPSG:3857 (cached), mosaicked; alpha < 1 = faded
    layer = url.split("/")[4].split(".")[-1]
    size = 2 * ORIGIN / 2 ** zoom
    tx0, tx1 = int((xmin + ORIGIN) // size), int((xmax + ORIGIN) // size)
    ty0, ty1 = int((ORIGIN - ymax) // size), int((ORIGIN - ymin) // size)
    rows = []
    for y in range(ty0, ty1 + 1):
        row = []
        for x in range(tx0, tx1 + 1):
            path = MAP_CACHE / "tiles" / f"{layer}_{zoom}_{x}_{y}.jpeg"
            if not path.exists():
                download(url.format(z=zoom, x=x, y=y), path)
            row.append(plt.imread(path))
        rows.append(np.hstack(row))
    img = np.vstack(rows)
    ax.imshow(img, extent=(tx0 * size - ORIGIN, (tx1 + 1) * size - ORIGIN, ORIGIN - (ty1 + 1) * size, ORIGIN - ty0 * size),
              cmap="gray" if img.ndim == 2 else None, interpolation="lanczos", alpha=alpha)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)


def label(ax, x, y, text, key, **kwargs):
    # Bold label with a white halo beside a point, placed by LABEL_OFFSET
    dx, dy = LABEL_OFFSET.get(key, (6, 4))
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", ha="right" if dx < 0 else "left",
                va="top" if dy < 0 else "baseline", fontweight="bold", path_effects=HALO, zorder=4, **kwargs)


def map_furniture(ax, scale_km, credit):
    # Scale bar (Web Mercator stretches distances by 1 / cos(lat)), north arrow and credit; no axes
    (x0, x1), (y0, y1) = ax.get_xlim(), ax.get_ylim()
    bar = scale_km * 1000 / np.cos(np.radians(AOI_LAT))
    bx, by = x0 + 0.05 * (x1 - x0), y0 + 0.04 * (y1 - y0)
    ax.plot([bx, bx + bar], [by, by], color="0.1", lw=3, solid_capstyle="butt")
    ax.text(bx + bar / 2, by + 0.015 * (y1 - y0), f"{scale_km} km", ha="center", fontsize=9, color="0.1",
            path_effects=HALO)
    ax.annotate("N", xy=(0.95, 0.95), xytext=(0.95, 0.87), xycoords="axes fraction", ha="center", fontsize=11,
                fontweight="bold", arrowprops=dict(arrowstyle="-|>", color="0.1"), path_effects=HALO)
    ax.text(0.99, 0.005, credit, transform=ax.transAxes, ha="right", va="bottom", fontsize=7, color="0.3",
            path_effects=HALO)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")


HALO = [patheffects.withStroke(linewidth=3, foreground="white")]
to_mercator = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

# AOI: the larch pixels (band 2, elevation band, is set on every AOI pixel) as polygons
aoi_mask = zones_tif.isel(band=1).notnull().values
aoi = gpd.GeoSeries([shape(g) for g, _ in shapes(aoi_mask.astype("uint8"), mask=aoi_mask,
                                                 transform=zones_tif.rio.transform())], crs=zones_tif.rio.crs)
aoi_elev = float(zones_tif.isel(band=0).values[aoi_mask].mean())

# Swiss National Park border (BAFU), drawn on both maps
SNP_COLLECTION = "ch.bafu.schutzgebiete-schweizerischer_nationalpark"
SNP_COLOR = "#1b7837"
snp_zip = MAP_CACHE / "nationalpark.shp.zip"
if not snp_zip.exists():
    snp_assets = stac_items(SNP_COLLECTION)[0]["assets"].values()
    download(next(a["href"] for a in snp_assets if a["href"].endswith(".shp.zip")), snp_zip)
snp = gpd.read_file(snp_zip)

# ---- Stations around the AOI, with the search radius (geodesic circle around the AOI centre)
lons, lats, _ = Geod(ellps="WGS84").fwd(np.full(361, AOI_LON), np.full(361, AOI_LAT), np.arange(361),
                                        np.full(361, VALIDATION_RADIUS_KM * 1000))
cx, cy = to_mercator.transform(lons, lats)
pad = 0.04 * (cx.max() - cx.min())

fig, ax = plt.subplots(figsize=(8, 8))
basemap(ax, cx.min() - pad, cy.min() - pad, cx.max() + pad, cy.max() + pad, zoom=12)
ax.plot(cx, cy, "--", color="0.2", lw=1.2)
snp.to_crs("EPSG:3857").plot(ax=ax, facecolor=SNP_COLOR, alpha=0.08)
snp.to_crs("EPSG:3857").boundary.plot(ax=ax, color=SNP_COLOR, lw=1.5)
aoi.to_crs("EPSG:3857").plot(ax=ax, color=LARCH_COLOR, edgecolor=LARCH_COLOR, lw=0.8)
sx, sy = to_mercator.transform(near.lon.values, near.lat.values)
ax.scatter(sx, sy, s=70, color=STATION_COLOR, edgecolor="white", lw=1.2, zorder=3)
for x, y, row in zip(sx, sy, near.itertuples()):
    label(ax, x, y, f"{row.name}\n{row.height_masl:.0f} m", row.Index, fontsize=9, color="0.1")
lx, ly = to_mercator.transform(local_sites.lon.values, local_sites.lat.values)
ax.scatter(lx, ly, s=55, marker="D", color=LOCAL_COLOR, edgecolor="white", lw=1.2, zorder=3.5)
for x, y, row in zip(lx, ly, local_sites.itertuples()):
    label(ax, x, y, f"{row.name}\n{row.height_masl:.0f} m", row.Index, fontsize=8, color=LOCAL_COLOR)
map_furniture(ax, 10, "Map: © swisstopo | Stations: MeteoSwiss | National Park: BAFU")
ax.legend(handles=[Patch(color=LARCH_COLOR, label=f"AOI larch stands (mean {aoi_elev:.0f} m)"),
                   Line2D([], [], ls="", marker="o", ms=8, color=STATION_COLOR, mec="white",
                          label="MeteoSwiss larch phenology station"),
                   Line2D([], [], ls="", marker="D", ms=7, color=LOCAL_COLOR, mec="white",
                          label="local larch phenology site (Lärchen_pheno)"),
                   Patch(facecolor=to_rgba(SNP_COLOR, 0.08), edgecolor=SNP_COLOR, lw=1.5, label="Swiss National Park"),
                   Line2D([], [], ls="--", color="0.2", label=f"search radius {VALIDATION_RADIUS_KM} km")],
          loc="upper left", fontsize=9, framealpha=0.9)
ax.set_title("Larch phenology stations and sites around the AOI")
plt.tight_layout()
fig.savefig(VALIDATION_DIR / "station_map.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ---- The AOI and the Swiss National Park
inside = aoi.intersection(snp.to_crs(aoi.crs).union_all()).area.sum() / aoi.area.sum()   # share of the larch area

aoi_m, snp_m = aoi.to_crs("EPSG:3857"), snp.to_crs("EPSG:3857")
xmin, ymin, xmax, ymax = np.r_[np.minimum(aoi_m.total_bounds[:2], snp_m.total_bounds[:2]),
                       np.maximum(aoi_m.total_bounds[2:], snp_m.total_bounds[2:])]
pad = 0.06 * max(xmax - xmin, ymax - ymin)
fig, ax = plt.subplots(figsize=(8, 8 * (ymax - ymin + 2 * pad) / (xmax - xmin + 2 * pad)))
basemap(ax, xmin - pad, ymin - pad, xmax + pad, ymax + pad, zoom=13)
snp_m.plot(ax=ax, facecolor=SNP_COLOR, alpha=0.08)
snp_m.boundary.plot(ax=ax, color=SNP_COLOR, lw=2)
aoi_m.plot(ax=ax, color=LARCH_COLOR, edgecolor=LARCH_COLOR, lw=0.5)
map_furniture(ax, 5, "Map: © swisstopo | National Park: BAFU")
ax.legend(handles=[Patch(color=LARCH_COLOR, label=f"AOI larch stands ({inside:.0%} inside the park)"),
                   Patch(facecolor=to_rgba(SNP_COLOR, 0.08), edgecolor=SNP_COLOR, lw=2,
                         label="Swiss National Park")],
          loc="upper left", fontsize=9, framealpha=0.9)
ax.set_title("AOI larch stands and the Swiss National Park")
plt.tight_layout()
fig.savefig(VALIDATION_DIR / "aoi_map.png", dpi=300, bbox_inches="tight")
plt.show()

## Checks of the larch classification (AOI)

Two maps to judge the larch pixels of the AOI themselves:

- **`aoi_reference_map.png`: forest-type map.** The AOI against the polygons of `REFERENCE_FOREST_FILE` (`Lärchen_SNP.shp`, forest type in `Forest_typ`). The AOI pixels are outlined: orange in a larch polygon, blue in a polygon of another forest type, black outside the mapped forest. The forest-type map is filled: larch light green, the other forest types grey. Top left: the whole AOI; the other panels: close-ups of 2 × `DETAIL_HALF_M` around the local sites `DETAIL_SITES` (Il Fuorn, Val Cluozza, Val Trupchun). The Il Fuorn box is also on the image map. The table `aoi_forest_types.csv` gives the AOI area per forest type. In the other direction, the share of the map's larch polygons inside the National Park that the AOI covers. The AOI lies entirely inside the park, so this share is taken inside the park.
- **`aoi_imagery_map.png`: images.** The outline of the AOI drawn over SWISSIMAGE (swisstopo aerial orthophoto, 10 cm, mostly flown in summer) and over a Sentinel-2 true-colour scene from mid-October, when larch is yellow and spruce, pine and Swiss stone pine stay dark green. Top: the whole AOI; bottom: the close-up around the first of `DETAIL_SITES` (Il Fuorn). The Sentinel-2 scene is the clearest cached date (whole AOI covered, at least `AUTUMN_IMAGE_MIN_CLEAR` clear, no snow) closest to `AUTUMN_IMAGE_DOY`.

In [ ]:
REF_TYPE = "Forest_typ"   # forest-type column of REFERENCE_FOREST_FILE
REF_LARCH = "Larch"       # its larch class
REF_LARCH_COLOR, REF_OTHER_COLOR = "#a6d96a", "#bdbdbd"
OTHER_TYPE_COLOR, NO_FOREST_COLOR = "#2166ac", "black"

ref = gpd.read_file(REFERENCE_FOREST_FILE).to_crs(aoi.crs)
ref["geometry"] = ref.geometry.force_2d().buffer(0)
ref = ref[[REF_TYPE, "geometry"]]
parts = gpd.overlay(gpd.GeoDataFrame(geometry=aoi), ref, how="intersection", keep_geom_type=True)
outside = gpd.GeoSeries([aoi.union_all().difference(ref.union_all())], crs=aoi.crs)

# ---- AOI area per forest type, and the park's larch polygons covered by the AOI
area = parts.area.groupby(parts[REF_TYPE]).sum()
area["not in the forest-type map"] = outside.area.sum()
aoi_types = pd.DataFrame({"area_ha": area / 1e4, "share_of_aoi": area / aoi.area.sum()}).sort_values("area_ha",
                                                                                                    ascending=False)
aoi_types.rename_axis("forest_type").to_csv(VALIDATION_DIR / "aoi_forest_types.csv")
display(aoi_types.round(3))
is_larch = parts[REF_TYPE] == REF_LARCH
park_larch = ref[ref[REF_TYPE] == REF_LARCH].clip(snp.to_crs(aoi.crs))
covered = parts[is_larch].area.sum() / park_larch.area.sum()
in_larch = aoi_types.share_of_aoi.get(REF_LARCH, 0)
print(f"AOI in {REF_LARCH} polygons: {in_larch:.0%} of its area. {REF_LARCH} polygons inside the National Park "
      f"({park_larch.area.sum() / 1e4:.0f} ha) covered by the AOI: {covered:.0%}")

# ---- close-ups: boxes around local sites (the first one also on the image map below)
DETAIL_SITES, DETAIL_HALF_M = ["fuo", "clu", "tru"], 1000   # box of 2 x DETAIL_HALF_M around each site
DETAIL_KM = f"{2 * DETAIL_HALF_M / 1000:.0f} × {2 * DETAIL_HALF_M / 1000:.0f} km"


def detail_box(code):
    # Box around a local site in EPSG:3857 (x0, y0, x1, y1) and the site's position
    s = local_sites.loc[code]
    x, y = to_mercator.transform(s.lon, s.lat)
    h = DETAIL_HALF_M / np.cos(np.radians(s.lat))   # Web Mercator stretch
    return (x - h, y - h, x + h, y + h), (x, y)


site = local_sites.loc[DETAIL_SITES[0]]
detail, (dx, dy) = detail_box(DETAIL_SITES[0])
half = (detail[2] - detail[0]) / 2

# ---- map: whole AOI and detail; AOI pixels and larch polygons as outlines
aoi_m = aoi.to_crs("EPSG:3857")
xmin, ymin, xmax, ymax = aoi_m.total_bounds
pad = 0.06 * max(xmax - xmin, ymax - ymin)
overview = (xmin - pad, ymin - pad, xmax + pad, ymax + pad)
ref_m = ref.to_crs("EPSG:3857")
ref_larch = ref_m[REF_TYPE] == REF_LARCH
aoi_classes = [(g.to_crs("EPSG:3857").boundary, color) for g, color in
               [(parts[is_larch], LARCH_COLOR), (parts[~is_larch], OTHER_TYPE_COLOR), (outside, NO_FOREST_COLOR)]]
panels = [("whole AOI", overview, None, 13, 0.4, 5)]   # title, box, site position, tile zoom, line width, scale bar km
panels += [(f"around {local_sites.name[code]} ({DETAIL_KM})", *detail_box(code), 16, 1.2, 0.5) for code in DETAIL_SITES]
nrows = int(np.ceil(len(panels) / 2))
fig, axes = plt.subplots(nrows, 2, figsize=(16, 8 * nrows), squeeze=False)
for ax in axes.flat[len(panels):]:
    ax.set_axis_off()
for ax, (title, box, xy, zoom, lw, scale) in zip(axes.flat, panels):
    basemap(ax, *box, zoom=zoom)
    ref_m[~ref_larch].plot(ax=ax, color=REF_OTHER_COLOR, alpha=0.6, lw=0)
    ref_m[ref_larch].plot(ax=ax, color=REF_LARCH_COLOR, alpha=0.8, lw=0)
    snp.to_crs("EPSG:3857").boundary.plot(ax=ax, color=SNP_COLOR, lw=1.5)
    for outline, color in aoi_classes:
        outline.plot(ax=ax, color=color, lw=lw)
    map_furniture(ax, scale, f"Map: © swisstopo | Forest types: {REFERENCE_FOREST_FILE.name} | National Park: BAFU")
    ax.set_title(title)
    if xy is not None:
        ax.plot(*xy, "D", ms=8, color=LOCAL_COLOR, mec="white", mew=1.2)
for code in DETAIL_SITES:   # close-up boxes on the overview
    (x0, y0, x1, y1), _ = detail_box(code)
    axes[0, 0].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="black", lw=1.5))
    axes[0, 0].text(x0, y1, local_sites.name[code], ha="left", va="bottom", fontsize=9, fontweight="bold",
                    path_effects=HALO)
other_share = aoi_types.share_of_aoi.drop([REF_LARCH, "not in the forest-type map"], errors="ignore").sum()
fig.legend(handles=[Patch(facecolor="none", edgecolor=LARCH_COLOR, lw=1.5,
                          label=f"AOI in a {REF_LARCH.lower()} polygon ({in_larch:.0%})"),
                    Patch(facecolor="none", edgecolor=OTHER_TYPE_COLOR, lw=1.5,
                          label=f"AOI in another forest type ({other_share:.0%})"),
                    Patch(facecolor="none", edgecolor=NO_FOREST_COLOR, lw=1.5,
                          label=f"AOI outside the mapped forest ({aoi_types.share_of_aoi['not in the forest-type map']:.0%})"),
                    Patch(color=REF_LARCH_COLOR, alpha=0.8,
                          label=f"{REF_LARCH.lower()} polygons of the forest-type map ({covered:.0%} of them in the park "
                                "covered by the AOI)"),
                    Patch(color=REF_OTHER_COLOR, alpha=0.6, label="other forest types"),
                    Patch(facecolor="none", edgecolor=SNP_COLOR, lw=1.5, label="Swiss National Park"),
                    Line2D([], [], ls="", marker="D", ms=7, color=LOCAL_COLOR, mec="white", label="local site")],
           loc="lower center", ncol=4, fontsize=9)
fig.suptitle("AOI larch pixels vs the forest-type map")
plt.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(VALIDATION_DIR / "aoi_reference_map.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
SWISSIMAGE = "https://wmts.geo.admin.ch/1.0.0/ch.swisstopo.swissimage/default/current/3857/{z}/{x}/{y}.jpeg"
AUTUMN_IMAGE_DOY = 286        # mid-October, about the whole-AOI peak of the yellowing (04)
AUTUMN_IMAGE_MIN_CLEAR = 0.7  # clear share of the AOI the scene needs

# ---- Sentinel-2 scene: the clearest cached date near AUTUMN_IMAGE_DOY
f = clean_frame(load_stats(), "grvi")
f = f[(f.covered_fraction >= 0.99) & (f.valid_fraction >= AUTUMN_IMAGE_MIN_CLEAR) & (f.scl_snow_fraction < 0.01)]
f = f.assign(days_off=abs(f.index.dayofyear - AUTUMN_IMAGE_DOY)).sort_values(["days_off", "valid_fraction"],
                                                                             ascending=[True, False])
day = f.index[0]
print(f"Sentinel-2 scene: {day:%d %b %Y} (DOY {day.dayofyear}), {f.valid_fraction.iloc[0]:.0%} of the AOI clear")
catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1",
                                    modifier=planetary_computer.sign_inplace)
xmin, ymin, xmax, ymax = aoi_m.total_bounds
pad = 0.03 * max(xmax - xmin, ymax - ymin)
overview = (xmin - pad, ymin - pad, xmax + pad, ymax + pad)
to_wgs84_m = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
box_wgs84 = to_wgs84_m.transform_bounds(*overview)
items = catalog.search(collections=["sentinel-2-l2a"], bbox=box_wgs84, datetime=f"{day:%Y-%m-%d}").item_collection()
epsg = aoi.crs.to_epsg()
rgb = stackstac.stack(items, assets=["B04", "B03", "B02"], resolution=10, epsg=epsg,
                      bounds_latlon=box_wgs84, resampling=0, xy_coords="center")
offset = xr.where(rgb["s2:processing_baseline"].astype(float) >= 4, 1000, 0)   # see 01, section 4
rgb = (rgb - offset).clip(min=0).mean("time").compute()
# Contrast stretch per band between its 2nd and 98th percentile (bright rock and snow saturate), slightly brightened
lo = rgb.quantile(0.02, dim=["x", "y"]).drop_vars("quantile")
hi = rgb.quantile(0.98, dim=["x", "y"]).drop_vars("quantile")
rgb = ((rgb - lo) / (hi - lo)).clip(0, 1) ** 0.7
rgb = rgb.rio.write_crs(epsg).rio.reproject("EPSG:3857", nodata=np.nan)
rgb_extent = (*rgb.rio.bounds()[0::2], *rgb.rio.bounds()[1::2])   # left, right, bottom, top

fig, axes = plt.subplots(2, 2, figsize=(14, 13.5))
for row, (box, zoom, lw, scale) in enumerate([(overview, 13, 0.4, 5), (detail, 17, 1.0, 0.5)]):
    for col, image in enumerate(["SWISSIMAGE", "Sentinel-2"]):
        ax = axes[row, col]
        if image == "SWISSIMAGE":
            basemap(ax, *box, zoom=zoom, url=SWISSIMAGE, alpha=1)
        else:
            ax.imshow(np.moveaxis(rgb.values, 0, -1), extent=rgb_extent, interpolation="nearest")
            ax.set_xlim(box[0], box[2])
            ax.set_ylim(box[1], box[3])
        aoi_m.boundary.plot(ax=ax, color=LARCH_COLOR, lw=lw)
        if row == 0:
            ax.add_patch(Rectangle(detail[:2], 2 * half, 2 * half, fill=False, edgecolor="white", lw=1.5))
        else:
            ax.plot(dx, dy, "D", ms=8, color=LOCAL_COLOR, mec="white", mew=1.2)
        map_furniture(ax, scale, f"{'© swisstopo' if image == 'SWISSIMAGE' else 'Contains Copernicus data'}")
        where = "whole AOI" if row == 0 else f"around {site['name']} ({2 * DETAIL_HALF_M / 1000:.0f} × {2 * DETAIL_HALF_M / 1000:.0f} km)"
        what = "SWISSIMAGE (aerial, 10 cm)" if image == "SWISSIMAGE" else f"Sentinel-2 true colour, {day:%d %b %Y}"
        ax.set_title(f"{what}: {where}", fontsize=10)
fig.legend(handles=[Patch(facecolor="none", edgecolor=LARCH_COLOR, lw=1.5, label="AOI larch pixels (outline)"),
                    Line2D([], [], ls="", marker="D", ms=7, color=LOCAL_COLOR, mec="white", label="local site")],
           loc="lower center", ncol=2, fontsize=9)
fig.suptitle("AOI larch pixels over images")
plt.tight_layout(rect=(0, 0.03, 1, 1))
fig.savefig(VALIDATION_DIR / "aoi_imagery_map.png", dpi=300, bbox_inches="tight")
plt.show()